# ACI Pro-Forma Workbench — Project 1

**Decision supported:** should the fund *initiate*, *watch/defer*, or *not initiate* a position in Albertsons Companies, Inc. (NYSE: ACI)?
**User:** the fund's investment committee (no current position). **Horizon:** through the FY2026 Q4 call and turnaround plan (spring 2027).
**Valuation date / bridge balance sheet:** Sept 12, 2026 (Q2 FY2026). **Units:** USD millions except per share. **Share basis:** 488.2m diluted (Q2 FY2026).

How to use: run all cells. The **driver panel** (cell 3) moves bounded drivers; **Reset to base** restores them. The **checks panel** turns red when an input combination breaks the model or leaves the historical band. Every number is computed by `analysis.py`; this notebook only displays it.


In [1]:
import json
from dataclasses import replace, asdict
from pathlib import Path
import analysis as A
from IPython.display import display, HTML

def table(rows: dict, fmt="{:,.1f}") -> HTML:
    # Render {row_label: {column: value}} as an HTML table (no pandas needed)
    cols = list(next(iter(rows.values())).keys())
    head = "".join(f"<th style='text-align:right;padding:2px 10px'>{c}</th>" for c in cols)
    body = "".join("<tr><td style='padding:2px 10px'>" + str(k) + "</td>" +
                   "".join(f"<td style='text-align:right;padding:2px 10px'>{fmt.format(v)}</td>" for v in r.values()) + "</tr>"
                   for k, r in rows.items())
    return HTML(f"<table><tr><th></th>{head}</tr>{body}</table>")
BASE_RESULT = A.run(A.BASE)
BASE_VALUE = BASE_RESULT["valuation"]["value_per_share"]
print(f"Engine loaded. Base value per share: ${BASE_VALUE:.2f}")

Engine loaded. Base value per share: $17.89


## 1. Driver panel (semi-adjustable: drivers exposed, engine and checks locked)

In [2]:
import ipywidgets as w

LABELS = {"growth_fy2026": "Revenue growth FY26", "growth_long": "Revenue growth FY28-30",
          "gross_margin_fy2026": "Gross margin FY26", "gross_margin_long": "Gross margin FY30",
          "sga_to_gp_fy2026": "SG&A / gross profit FY26", "sga_to_gp_long": "SG&A / gross profit FY30",
          "capex": "Capex ($m / yr)", "wacc": "WACC", "terminal_growth": "Terminal growth"}
sliders = {}
for k, (lo, hi, blo, bhi, unit, basis) in A.BOUNDS.items():
    step = 10.0 if unit == "$m" else 0.0005
    fmt = ".0f" if unit == "$m" else ".2%"
    sliders[k] = w.FloatSlider(value=getattr(A.BASE, k), min=lo, max=hi, step=step, readout_format=fmt,
                               description=LABELS[k], style={"description_width": "190px"},
                               layout=w.Layout(width="520px"), continuous_update=False,
                               tooltip=basis)
reset = w.Button(description="Reset to base", button_style="info")
out = w.Output()

def render(*_):
    d = replace(A.BASE, **{k: s.value for k, s in sliders.items()})
    with out:
        out.clear_output(wait=True)
        try:
            r = A.run(d)
        except Exception as e:  # engine refuses incoherent inputs loudly
            display(HTML(f"<div style='background:#fbeae0;color:#a4400f;padding:8px'><b>MODEL REFUSED INPUT:</b> {e}</div>"))
            return
        v, s = r["valuation"], r["statements"]
        rows = []
        for name, ok, detail in r["checks"]:
            color = "#1f6b4f" if ok else "#c0392b"
            rows.append(f"<tr><td style='color:{color};font-weight:700'>{'PASS' if ok else 'FAIL'}</td><td>{name}</td><td>{detail}</td></tr>")
        n_fail = sum(not ok for _, ok, _ in r["checks"])
        banner = ("<div style='background:#c0392b;color:white;padding:6px 10px;font-weight:700'>"
                  f"{n_fail} CHECK(S) FAILING — this input combination is outside what the model can defend</div>") if n_fail else ""
        if v:
            delta = v["value_per_share"] - BASE_VALUE
            display(HTML(f"{banner}<h3 style='margin:6px 0'>Value per share ${v['value_per_share']:.2f} "
                         f"<span style='font-weight:400'>({delta:+.2f} vs base ${BASE_VALUE:.2f})</span></h3>"
                         f"EV ${v['enterprise_value']:,.0f}m − debt ${v['debt']:,.0f}m − opioid obligation ${v['other_debt_like']:,.0f}m + cash ${v['cash']:,.0f}m = equity ${v['equity_value']:,.0f}m "
                         f"÷ {v['shares']:.1f}m shares · terminal value {v['terminal_share']:.1%} of EV"))
        display(table({k: {f"FY{y}E": s[y][k] for y in A.YEARS} for k in ("revenue", "ebitda", "ebit", "net_income", "fcff", "cash", "bs_gap")}))
        display(HTML("<table>" + "".join(rows) + "</table>"))

def do_reset(_):
    for k, s in sliders.items():
        s.value = getattr(A.BASE, k)
    render()

for s in sliders.values():
    s.observe(render, names="value")
reset.on_click(do_reset)
display(w.VBox([w.HBox([reset]), *sliders.values(), out]))
render()

## 2. Frozen base-case output (visible without running)

In [3]:
r = BASE_RESULT
s, v = r["statements"], r["valuation"]
lines = ["revenue","gross_profit","sga","ebitda","ebit","interest","net_income",
         "inventory","accounts_payable","ppe","cash","debt","equity","total_assets","bs_gap",
         "cfo","cfi","cff","net_change_in_cash","nopat","da","capex","delta_nwc","fcff","fcfe"]
display(table({k: {f"FY{y}E": s[y][k] for y in A.YEARS} for k in lines}))
print(f"EV ${v['enterprise_value']:,.1f}m | equity ${v['equity_value']:,.1f}m | ${v['value_per_share']:.2f}/share | TV share {v['terminal_share']:.1%}")

,FY2026E,FY2027E,FY2028E,FY2029E,FY2030E
revenue,"81,403.4","81,810.5","83,037.6","84,283.2","85,547.4"
gross_profit,"21,816.1","21,966.1","22,337.1","22,714.3","23,097.8"
sga,"20,507.2","20,593.2","20,885.2","21,181.1","21,481.0"
ebitda,"3,222.7","3,303.3","3,395.7","3,487.9","3,580.3"
ebit,"1,309.0","1,372.9","1,451.9","1,533.2","1,616.8"
interest,527.8,524.9,521.9,519.0,516.0
net_income,581.9,631.7,692.8,755.6,820.1
inventory,"5,093.5","5,115.5","5,188.6","5,262.9","5,338.2"
accounts_payable,"3,956.2","3,973.3","4,030.1","4,087.8","4,146.3"
ppe,"9,990.0","10,059.6","10,115.8","10,161.1","10,197.7"


EV $18,553.2m | equity $8,733.2m | $17.89/share | TV share 79.7%


### Causal cases, WACC × growth grid, one-at-a-time sensitivity

In [4]:
print("Cases ($/share):")
for k, x in A.cases().items():
    print(f"  {k:<42} ${x:6.2f}")
grid = A.wacc_growth_grid()
display(table({f"WACC {w_:.1%}": {f"g {g:.1%}": x for g, x in row.items()} for w_, row in grid.items()}, "${:.2f}"))
display(table({f"{t['driver']} ({t['low']:.4g} to {t['high']:.4g})": {"value at low": t["at_low"]["value_per_share"],
               "value at high": t["at_high"]["value_per_share"], "swing $/share": t["swing"]} for t in A.tornado()}, "${:.2f}"))

Cases ($/share):
  Low: comps stay negative, margin slips     $  7.92
  Base                                       $ 17.89
  High: turnaround works by FY2028           $ 24.74


,g 1.0%,g 1.5%,g 2.0%,g 2.5%
WACC 6.7%,$19.06,$22.24,$26.09,$30.87
WACC 7.2%,$15.85,$18.47,$21.60,$25.39
WACC 7.7%,$13.11,$15.31,$17.89,$20.97
WACC 8.2%,$10.76,$12.62,$14.78,$17.32
WACC 8.7%,$8.71,$10.30,$12.14,$14.26


,value at low,value at high,swing $/share
sga_to_gp_long (0.904 to 0.96),$31.41,$2.28,$29.13
wacc (0.07 to 0.095),$23.29,$8.64,$14.65
terminal_growth (0.01 to 0.025),$13.11,$20.97,$7.85
capex (1800 to 2200),$21.11,$14.67,$6.45
sga_to_gp_fy2026 (0.904 to 0.96),$19.99,$16.72,$3.27
gross_margin_long (0.264 to 0.279),$17.00,$19.22,$2.22
growth_long (0 to 0.025),$16.63,$18.75,$2.12
growth_fy2026 (-0.015 to 0.02),$17.32,$19.32,$1.99
gross_margin_fy2026 (0.264 to 0.279),$17.88,$17.91,$0.03


### Checks

In [5]:
for n, ok, x in BASE_RESULT["checks"] + A.base_year_reconciliation():
    print(("PASS " if ok else "FAIL ") + n + " — " + x)

PASS Balance sheet balances every year — max |gap| = 0.000000
PASS Cash flow ties to balance-sheet cash — max |difference| = 0.000000
PASS Cash stays at or above $100m minimum — lowest cash = 329.7
PASS Revolver within $3,562m limit — peak revolver = 0.0
PASS Terminal growth below WACC — g 2.00% vs WACC 7.70%
PASS Drivers inside historical band — all inside
PASS FY2026E EBITDA within 15% of H1 FY2026 run-rate — model 3,223 vs run-rate 3,337 (-3.4%)
PASS Enterprise-to-equity bridge reconciles — EV 18,553 - debt 9,328 - opioid 774 + cash 282 = 8,733
PASS Terminal value below 85% of EV — TV share 79.7%
PASS Normalized revenue = filed revenue - 53rd week — 81,812.5 + 1,360.0 = 83,172.5
PASS Normalized SG&A/GP excludes opioid charge — 93.41% of gross profit
PASS FY2025 opening balance sheet balances — assets 26,765.9 vs liabilities+equity 26,765.9


## 3. Export the frozen visible output
Uses the dated, sourced quote in `analysis.MARKET`.

In [6]:
PRICE = A.MARKET["price"]            # $13.09, ACI close Oct 16, 2026 (source in docs ledger)
PRICE_DATE = A.MARKET["price_date"]
summary = A.summary(PRICE, PRICE_DATE)
Path("outputs").mkdir(exist_ok=True)
Path("outputs/visible_output.json").write_text(json.dumps(summary, indent=2, default=float))
print("Wrote outputs/visible_output.json")

Wrote outputs/visible_output.json
